# Environmental controls of measured tilt
Environmental PV-gradient magnitude, planetary beta, latitude, bathymetry, gradient coherence and existing background shear. PV magnitudes use **log10** for display and comparison. The standard environmental gradient excludes the internal eddy-vorticity-gradient contribution. Core stratification is analysed in the companion eddy-properties notebook, not relabelled as background stratification.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd().resolve()
ANALYSIS=next((p for p in (HERE,*HERE.parents) if (p/'seacofs_tilt_tools.py').exists()),None)
if ANALYSIS is None: raise FileNotFoundError('Launch inside seacofs_eddy_tilt_analysis')
WORK=ANALYSIS/'eddy_env_tilt_controls'
for p in (ANALYSIS,WORK,ANALYSIS/'tilt_mechanisms'):
    if str(p) not in sys.path: sys.path.insert(0,str(p))
import seacofs_tilt_tools as tilt
import control_tools as ctl
import mechanism_tools as mech
N_BOOT=300
MIN_EDDIES_PER_BIN=10
MAX_DEPTH_M=1000
TARGET_DEPTHS=[0,100,200,500,1000]
DEPTH_TOLERANCE_M=65
MIN_N2_COVERAGE=.8
SAVE_RESULTS=True
paths=tilt.Paths()
OUTPUT_ROOT=Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/eddy_env_tilt_controls')
N2_CACHE=mech.DEFAULT_N2_CACHE_PATH
VV_CACHE=Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/vertical_velocity_climatology/core_vertical_velocity_upper_1000m.parquet')
VV_FRACTION=1.5
MIN_VV_CELLS=8
MIN_VV_COVERAGE=.7
results={}
inputs=[paths.eddies,paths.tilt,paths.vert,paths.grid,paths.z_r,WORK/'control_tools.py']

In [ ]:
grid=tilt.load_grid(paths.grid,paths.z_r)
raw,_=tilt.load_tilt_tables(paths,add_regions=True,grid=grid)
surface=ctl.prepare_surface(raw,grid)
vertical=ctl.prepare_profiles(tilt.load_vert(paths),surface,MAX_DEPTH_M)
summary=ctl.profile_summaries(vertical)
surface=ctl.merge(surface,summary)
print('Surface rows:',len(surface),'Eddies:',surface.Eddy.nunique())
print('Rows with profiles:',surface.vertical_extent_m.notna().sum())
print('Measured TiltDis/TiltDir are loaded, never recomputed.')

## Reading the plots and statistics
Red = AE, blue = CE. Curves are **eddy-day medians** in shared quantile bins; shading is the IQR, not uncertainty. Vertical bars are pointwise 95% intervals obtained by resampling whole contributing eddies in each bin. Bins with fewer than `MIN_EDDIES_PER_BIN` distinct eddies are omitted. No bin, including the upper endpoint, is discarded.

An equal-eddy sensitivity computes correlations between per-eddy median properties and median tilt; its intervals resample eddies. These are between-eddy associations. The adjusted models use inverse-observation-count weights within each eddy and eddy-clustered standard errors, separately by polarity, with one focal property at a time. They control radius, vertical extent, latitude, region and season where available; a focal control is removed from its own control list. Latitude is omitted for beta because they encode the same geographic gradient. Estimates are changes in **log(1 + outcome)** per SD of the predictor. BH q-values cover the tested predictor/polarity family in each table. These exploratory models do not establish causation or remove all spatial dependence.

Profile depth coverage and the existing smoothed tilt estimator affect interpretation. Extrema grow sensitive to sampled depth range and sampling density. Compare the matched-depth analysis and common-depth profile summaries before interpreting full-profile maxima. `TiltDis/Rc` also shares radius with radius-related predictors; it is a sensitivity outcome, not independent confirmation.

## Existing surface PV cache
Preserve the established surface Gaussian footprint (`frac=1`, nonlinear averaging, signed largest-|vorticity| in the upper 1000 m). Cache validation checks these settings. The cache loader returns a saved table: only environmental fields are merged into the freshly loaded surface observations, preventing saved tilt/property values from replacing current data. Missing cache files stop with instructions; this notebook does not overwrite the shared cache.

Planetary beta is `df/dy` (m⁻¹ s⁻¹); the planetary PV-gradient contribution is beta/h (m⁻² s⁻¹). Logged gradients are log10 of numerical magnitudes in m⁻² s⁻¹. Nonpositive magnitudes remain missing. Latitude and beta are related geographic descriptions, not independent mechanisms. Coherence distinguishes weak mean vectors from cancellation among strong local gradients.

In [ ]:
inputs.append(tilt.DEFAULT_SURFACE_PV_CACHE)
if not tilt.DEFAULT_SURFACE_PV_CACHE.exists():
    raise FileNotFoundError(f'Build the established surface PV cache first: {tilt.DEFAULT_SURFACE_PV_CACHE}; see surface_pv_esp_gaussian.')
pv=tilt.add_pv_gradient_terms(surface,grid,core_mean=True,frac=1.,surface_method='esp_gaussian',averaging='nonlinear',use_max_abs_w=True,max_depth_m=1000,use_cache=True)
environment=ctl.add_environment(surface,pv)
env_metrics=['log10_PV_grad_mag','log10_PV_grad_plan_mag','log10_PV_grad_topo_mag',
             'log10_topo_plan_ratio','PV_grad_coherence','beta','lat','h','slope']
results['environment_availability']=ctl.audit(environment,env_metrics)
display(results['environment_availability'])
for outcome in ['TiltDis','tilt_over_radius']:
    results['environment_bins_'+outcome]=ctl.panels(environment,env_metrics,y=outcome,n_boot=N_BOOT,min_eddies=MIN_EDDIES_PER_BIN,title='Environmental controls')

## Background shear: existing cache only
This retains the agreed shear-magnitude and tilt-direction comparison. It does not add propagation or along-/cross-shear mechanism extensions. The centred 91-day climatological **surface-minus-0–500 m mean velocity difference** is a bulk shear proxy in m/s, not a local derivative in s⁻¹. The absolute angle from the deep-to-shallow tilt direction spans 0–180°. Zero shear or zero tilt has undefined alignment.

No background stratification estimate is available in the core-N2 cache; core N2 remains in the other notebook.

In [ ]:
from beta_effect_background_flow.background_flow_tools import BackgroundConfig,load_background_cache
bg_config=BackgroundConfig()
inputs.append(bg_config.background_table_path)
if bg_config.background_table_path.exists():
    background=load_background_cache(bg_config)
    environment=ctl.add_shear(environment,background)
    env_metrics+=['shear_mag_ms']
    results['shear_availability']=ctl.audit(environment,['shear_mag_ms','shear_angle_deg'])
    display(results['shear_availability'])
    results['shear_magnitude_bins']=ctl.panels(environment,['shear_mag_ms'],n_boot=N_BOOT,min_eddies=MIN_EDDIES_PER_BIN,title='Background shear magnitude')
    results['shear_direction_bins']=ctl.panels(environment,['shear_mag_ms'],y='shear_angle_deg',n_boot=N_BOOT,min_eddies=MIN_EDDIES_PER_BIN,title='Tilt direction relative to shear')
else:
    print('SHEAR SECTION UNAVAILABLE:',bg_config.background_table_path)

## Regional and seasonal sensitivity
Repeat key relationships within each region and season; tables retain counts so sparse groups remain apparent. Seasonal names denote calendar quarters (DJF/MAM/JJA/SON). These comparisons test population composition, not independent new replications.

In [ ]:
stratified_metrics=['log10_PV_grad_mag','beta','lat','h','slope']
if 'shear_mag_ms' in environment:stratified_metrics+=['shear_mag_ms']
for grouping in ['Region','Season']:
    if grouping not in environment:continue
    for label,part in environment.groupby(grouping,observed=True):
        key=f'{grouping}_{label}'
        results[key+'_availability']=ctl.audit(part,stratified_metrics)
        results[key+'_bins']=ctl.panels(part,stratified_metrics,n_boot=N_BOOT,min_eddies=MIN_EDDIES_PER_BIN,title=f'{grouping}: {label}')
        results[key+'_correlations']=ctl.correlations(part,stratified_metrics,n_boot=N_BOOT)

## Equal-eddy and adjusted comparisons
Each focal environmental predictor is fitted separately; correlated beta and latitude are not fitted together. Coefficients remain exploratory associations, including because the environmental topographic PV term itself depends on reconstructed eddy vorticity.

In [ ]:
for outcome in ['TiltDis','tilt_over_radius']:
    results['eddy_equal_correlations_'+outcome]=ctl.correlations(environment,env_metrics,y=outcome,n_boot=N_BOOT)
    results['adjusted_'+outcome]=ctl.adjusted(environment,env_metrics,y=outcome)
    display(results['eddy_equal_correlations_'+outcome])
    display(results['adjusted_'+outcome])

## Export tables and settings

In [ ]:
NOTEBOOK_NAME='environment'

if SAVE_RESULTS:
    settings=dict(n_boot=N_BOOT,min_eddies_per_bin=MIN_EDDIES_PER_BIN,max_depth_m=MAX_DEPTH_M,
                  target_depths=TARGET_DEPTHS,depth_tolerance_m=DEPTH_TOLERANCE_M,
                  n2_min_coverage=MIN_N2_COVERAGE,vv_fraction=VV_FRACTION,
                  vv_min_cells=MIN_VV_CELLS,vv_min_coverage=MIN_VV_COVERAGE,
                  unavailable_sections=[str(p) for p in inputs if not Path(p).exists()])
    saved=ctl.save_results(OUTPUT_ROOT/NOTEBOOK_NAME,results,settings,inputs)
    print('Saved tables and provenance to',saved)